# Deep Learning
## Summative assessment
### Coursework 1

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#### Instructions

This coursework is released on **12th February 9.00 (UK time)** and is due by **26th February 9.00 (UK time)**. It is worth **50%** of your overall mark. There are 4 questions in this assessment, and a total of 50 marks are available. **You should attempt to answer all questions.** In addition to the total number of marks shown per question below, an additional 5 marks is available for presentation and clarity/quality of code.

This assessment assesses your ability to design, implement, train and evaluate a deep learning model for a recommender system. This is the problem of recommending suitable items to a given user.

You can make imports as and when you need them throughout the notebook, and add code cells where necessary. Make sure your notebook executes correctly in sequence before submitting. You should use Keras for this assessment, only reverting to backend-specific implementations where necessary.

The data required for this assessment can also be downloaded from [this link](https://drive.google.com/file/d/1Px16gtpLczdnpTmewBSnhKL_cIAY22w7/view?usp=sharing) in case you would prefer to work locally.

#### Submission instructions

The submission for this assessment will consist of a notebook (.ipynb file) and PDF report.

Ensure your notebook executes correctly in order. Save your notebook .ipynb file **after you have executed it** (so that outputs are all showing). It is recommended to also submit a PDF/HTML copy of your executed notebook, in case the .ipynb file is corrupted for some reason.

Upload a zip file containing your notebook and separate PDF/HTML file(s) by the deadline above.

This assignment must be attempted individually; your submission must be your own, unaided work. Candidates are prohibited from discussing assessed coursework, and must abide by [Imperial College’s rules](https://www.imperial.ac.uk/media/imperial-college/administration-and-support-services/registry/academic-governance/public/academic-policy/academic-integrity/Examination-and-assessments---academic-integrity.pdf) regarding academic integrity and plagiarism. Unless specifically authorised within the assignment instructions, the submission of output from [generative AI tools](https://www.imperial.ac.uk/about/leadership-and-strategy/provost/vice-provost-education/generative-ai-tools-guidance/) (e.g., ChatGPT) for assessed coursework is prohibited. Violations will be treated as an examination offence. Enabling other candidates to plagiarise your work constitutes an examination offence. To ensure quality assurance is maintained, departments may choose to invite a random selection of students to an ‘authenticity interview’ on their submitted assessments.

In [3]:
# You will need the following imports for this assessment. You can make additional imports when you need them

import keras
from keras import ops
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

from keras.models import Sequential, Model

#### YOOCHOOSE Dataset

This assessment makes use of the YOOCHOOSE dataset, which contains a collection of sessions from a retailer, where each session
consists of the item click events that a user performed in the session. The dataset is a standard benchmark dataset for the development of session-based recommendation algorithms. Session-based recommendation is where user IDs are not available, but sessions containing data about how users interact with a platform are available. The YOOCHOOSE dataset was created by YOOCHOOSE GmbH for the [2015 Recsys Challenge](https://recsys.acm.org/recsys15/challenge/).

The full dataset can be downloaded from [Kaggle](https://www.kaggle.com/datasets/chadgostopp/recsys-challenge-2015). It contains the data files `yoochoose-clicks.dat`, `yoochoose-test.dat` and `yoochoose-buys.dat`, as well as a `dataset-README.txt` file that describes the dataset. For this assessment only the data in `yoochoose-clicks.dat` will be used, and for this coursework a subset `yoochoose-clicks-subset.dat` has been prepared for you and is made available in the `./data` folder, as well as on Blackboard.

#### Recommender systems

Your task in this assessment is to develop a deep learning model that can be used to predict the next item that a user will interact with during a session of item click events. The model will take the sequence of items clicked on so far as the input, and will output a set of logits (that can be converted to probabilities using the softmax function) across the set of possible items.

In question 1 you will explore the dataset and prepare it for training your model. The training algorithm and model will be custom written for this task. In particular, the model will implement a particular form of weight sharing, and the training algorithm will employ custom logic that uses the specific training examples in a batch in order to reduce the computational load. In question 2 you will train and evaluate the specified model.

The model design and training algorithm will potentially require several hyperparameter choices, and in question 3 you should experiment with different choices during the development of your model, using a base model as a benchmark. It is up to you which experiments you will run. It is not necessary to include all experiments carried out in code in this notebook. You only need to include any custom code that you have written, and the code for the final choice of hyperparameters (in order to avoid overly long notebook submissions). In question 4 you will write a report detailing the experiments carried out.

### Question 1 (Total 10 marks)

a) In this question, you should load the `yoochoose-clicks-subset.dat` data into a pandas DataFrame, and carry out a basic exploration and analysis of the dataset. Compute, display and visualise any properties you deem to be relevant, along with some basic information on the click events, sessions and items. You can also use comments and/or markdown as appropriate.

The 'Category' column will not be used in this assessment, so can be ignored.

**(2 marks)**

In [4]:
df = pd.read_csv("/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/data/yoochoose-clicks-subset.dat", sep=",", header=None)

/tmp/ipython-input-3244742794.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/data/yoochoose-clicks-subset.dat", sep=",", header=None)


In [5]:
df.columns = ["Session ID", "Timestamp", "Item ID", "Category"]
df.drop(columns=["Category"], inplace=True)

In [6]:
num_unique_sessions = df['Session ID'].nunique()
num_unique_items = df['Item ID'].nunique()
pd.to_datetime(df['Timestamp'])
time_min = df['Timestamp'].min()
time_max = df['Timestamp'].max()
shape = df.shape

print(f"Number of unique Session IDs: {num_unique_sessions}")
print(f"Number of unique Item IDs: {num_unique_items}")
print(f"Time range: {time_min} to {time_max}")
print(f"The data size is {shape}")

df.head(30)

Number of unique Session IDs: 6012324
Number of unique Item IDs: 49499
Time range: 2014-04-01T03:00:00.124Z to 2014-09-30T02:59:59.430Z
The data size is (21452447, 3)


,Session ID,Timestamp,Item ID
0,1,2014-04-07T10:51:09.277Z,214536502
1,1,2014-04-07T10:54:09.868Z,214536500
2,1,2014-04-07T10:54:46.998Z,214536506
3,1,2014-04-07T10:57:00.306Z,214577561
4,3,2014-04-02T13:17:46.940Z,214716935
5,3,2014-04-02T13:26:02.515Z,214774687
6,3,2014-04-02T13:30:12.318Z,214832672
7,4,2014-04-07T12:09:10.948Z,214836765
8,4,2014-04-07T12:26:25.416Z,214706482
9,7,2014-04-02T06:38:53.104Z,214826835


b) Now you should preprocess the `yoochoose-clicks-subset.dat` data. The requirements are as follows:

* A test set should be created consisting of the sessions whose last click event takes place within the last 24 hours of the time period covered by the entire dataset. There should be no sessions that are split across data partitions.
* There should be no sessions consisting of a single click
* There should be no sessions longer than 10 click events
* Any items that appear in the test set but not the training set should be removed from the test set

You should define DataFrames for each of the data splits created. Print the resulting number of click events in each data split.

**(4 marks)**

In [7]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

max_time = df['Timestamp'].max()
cutoff_time = max_time - pd.Timedelta(days=1)

session_max_times = df.groupby('Session ID')['Timestamp'].max()

test_session_ids = session_max_times[session_max_times > cutoff_time].index
#print(test_session_ids)

train_session_ids = session_max_times[session_max_times <= cutoff_time].index
#print(train_session_ids)

train_df = df[df['Session ID'].isin(train_session_ids)]
test_df = df[df['Session ID'].isin(test_session_ids)]

In [8]:
train_session_lengths = train_df.groupby('Session ID').size()
test_session_lengths = test_df.groupby('Session ID').size()

valid_train_sessions = train_session_lengths[(train_session_lengths > 1) & (train_session_lengths <= 10)].index
valid_test_sessions = test_session_lengths[(test_session_lengths > 1) & (test_session_lengths <= 10)].index

train_df = train_df[train_df['Session ID'].isin(valid_train_sessions)].copy()
test_df = test_df[test_df['Session ID'].isin(valid_test_sessions)].copy()

In [9]:
train_df.head(10)

,Session ID,Timestamp,Item ID
0,1,2014-04-07 10:51:09.277000+00:00,214536502
1,1,2014-04-07 10:54:09.868000+00:00,214536500
2,1,2014-04-07 10:54:46.998000+00:00,214536506
3,1,2014-04-07 10:57:00.306000+00:00,214577561
4,3,2014-04-02 13:17:46.940000+00:00,214716935
5,3,2014-04-02 13:26:02.515000+00:00,214774687
6,3,2014-04-02 13:30:12.318000+00:00,214832672
7,4,2014-04-07 12:09:10.948000+00:00,214836765
8,4,2014-04-07 12:26:25.416000+00:00,214706482
9,7,2014-04-02 06:38:53.104000+00:00,214826835


In [10]:
test_df.head(10)

,Session ID,Timestamp,Item ID
20919625,11255568,2014-09-29 13:30:47.554000+00:00,214696432
20919626,11255568,2014-09-29 13:59:52.543000+00:00,214857030
20919630,11255571,2014-09-29 06:41:24.881000+00:00,214858854
20919631,11255571,2014-09-29 06:41:29.168000+00:00,214858854
20919632,11255572,2014-09-29 16:50:10.726000+00:00,214836819
20919633,11255572,2014-09-29 16:54:33.662000+00:00,214696434
20919669,11255599,2014-09-29 06:43:38.385000+00:00,214857570
20919670,11255599,2014-09-29 06:51:09.659000+00:00,214858847
20919671,11255599,2014-09-29 06:51:45.034000+00:00,214859094
20919672,11255599,2014-09-29 06:52:03.280000+00:00,214690730


In [11]:
train_items = set(train_df['Item ID'].unique())

initial_test_size = len(test_df)

test_df = test_df[test_df['Item ID'].isin(train_items)].copy()

dropped_count = initial_test_size - len(test_df)

print("\nFinal Dataset Statistics:")
print(f"Training Set: {len(train_df)} click events, {train_df['Session ID'].nunique()} unique sessions.")
print(f"Test Set:     {len(test_df)} click events, {test_df['Session ID'].nunique()} unique sessions.")
print(f"Dropped {dropped_count} interactions from Test Set containing unknown items.")


Final Dataset Statistics:
Training Set: 16597747 click events, 4949542 unique sessions.
Test Set:     33375 click events, 9258 unique sessions.
Dropped 39 interactions from Test Set containing unknown items.


c) You should now prepare TensorFlow Datasets or PyTorch DataLoaders (it is up to you which you choose) ready for training and evaluation.

The task of your model will be to predict the next item in the sequence, within a session. Therefore, your Datasets should return a tuple of Tensors `(inputs, outputs)`, such that both Tensors have shape `(batch_size, seq_len)`. The entries in the Tensors will correspond to item indices. As the sessions have varying length, you will need to make use of padding in the Dataset preparation. Your Datasets should use a batch size of 64.

You can use any tools from Keras, TensorFlow or PyTorch that might be useful for this task.

Demonstrate that your dataset creation and processing is working as intended by drawing a batch of examples from your dataset and displaying it.

**(4 marks)**

In [12]:
import tensorflow as tf

In [13]:
# We start indexing from 1, reserving 0 for padding
all_items = sorted(list(train_df['Item ID'].unique()))

item_to_index = {item: i+1 for i, item in enumerate(all_items)}
#print(item_to_index)

num_classes = len(all_items) + 1 # +1 for padding token 0

In [14]:
def create_sequences(df, mapping):
    df_sorted = df.sort_values(['Session ID', 'Timestamp'])
    df_sorted['Item Index'] = df_sorted['Item ID'].map(mapping)
    sequences = df_sorted.groupby('Session ID')['Item Index'].apply(list)
    return sequences

print("Creating sequences...")
train_sequences = create_sequences(train_df, item_to_index)
test_sequences = create_sequences(test_df, item_to_index)

print(f"First training sequence example: {train_sequences.iloc[0]}")

Creating sequences...
First training sequence example: [1941, 1940, 1942, 9210]


In [15]:
from keras.preprocessing.sequence import pad_sequences

def get_dataset_slices(sequence, batch_size= 64):
    inputs = []
    outputs = []
    for seq in sequence:
            inputs.append(seq[:-1])
            outputs.append(seq[1:])

    x_np = pad_sequences(inputs, padding='post', value=0)
    y_np = pad_sequences(outputs, padding='post', value=0)
    dataset = tf.data.Dataset.from_tensor_slices((x_np, y_np))


    dataset = dataset.shuffle(buffer_size=10000)
    dataset = dataset.batch(batch_size)

    return dataset

In [16]:
train_ds = get_dataset_slices(train_sequences)
test_ds = get_dataset_slices(test_sequences)

In [ ]:
def show_batch(dataset):
      for example in dataset.take(1):
          print(f"The batch sample is {example[0]}")
          print(f"The sample size is {example[0].shape}")



show_batch(train_ds)
show_batch(test_ds)

The batch sample is [[20212     0     0     0     0     0     0     0     0]
 [34114     0     0     0     0     0     0     0     0]
 [28346 28922 35216 28678 27731 34379 39756     0     0]
 [24008 23996 34352 34352  2587     0     0     0     0]
 [ 4069     0     0     0     0     0     0     0     0]
 [35291     0     0     0     0     0     0     0     0]
 [35338     0     0     0     0     0     0     0     0]
 [34370     0     0     0     0     0     0     0     0]
 [ 4992 28035 14727     0     0     0     0     0     0]
 [10454  1885  1887 10224  1877 36973 35378  1213     0]
 [34348 33916 18533 33917 34349 26867 34346     0     0]
 [39455     0     0     0     0     0     0     0     0]
 [ 3447     0     0     0     0     0     0     0     0]
 [20655 25106 12125  3696     0     0     0     0     0]
 [26863 10127 35256     0     0     0     0     0     0]
 [38930     0     0     0     0     0     0     0     0]
 [13693     0     0     0     0     0     0     0     0]
 [24549 245

### Question 2 (Total 15 marks)

a) You should now implement the model that you will train in the next-item prediction task. The specification for your model is as follows:

* The model should accept a batch of tokens with arbitrary sequence length
* The next layer is an embedding lookup layer with embedding dimension 64
* The next layer is a dropout layer with dropout rate 0.1
* There should then be a GRU layer with 64 hidden units. The GRU layer should return sequences so that the final output Tensor of the model has shape `(batch_size, seq_len, hidden_units)`

Print the model summary.

**(2 marks)**

In [35]:
from keras.layers import Embedding, Dropout, GRU, Dense, Input

In [ ]:
def get_model():

  model = Sequential()
  model.add(Input(shape=(None,)))
  model.add(Embedding(input_dim=num_classes, output_dim=64, mask_zero=True))
  model.add(Dropout(0.1))
  model.add(GRU(64, return_sequences=True))

  return model

In [ ]:
model = get_model()
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 64)       │     2,931,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, None, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, None, 64)       │        24,960 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,956,800 (11.28 MB)

 Trainable params: 2,956,800 (11.28 MB)

 Non-trainable params: 0 (0.00 B)

b) You should now implement and run a custom training loop for your model. The training algorithm is as follows:

* For each item prediction in the sequence, compute the logits by taking the inner product of the output of the GRU layer with the item embeddings stored in the model's embedding layer. In other words, you will re-use the embeddings in the (learned) embedding layer again at the output of the model to compute the logits. (This is why the hidden size of the GRU needed to match the embedding dimension.)
* Since there is a large number of possible items, you should only compute the logits with a subset of the items. The subset should consist of all of the items present in the current batch of examples.
* Given the computed logits over this subset of items, the loss should be computed as the categorical cross entropy loss across these items.

Your custom training loop should implement the above training algorithm. You should then run your custom training loop to train your model. It is recommended to save your model during the training in case of broken runs. Your custom training loop should periodically print out the loss during the training run to monitor performance.

**(8 marks)**

In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

In [40]:
import time
import tensorflow as tf

@tf.function
def train_step(inputs, targets, model, optimizer):
    with tf.GradientTape() as tape:

        # extract output and weight in embedding layer
        gru_output = model(inputs, training=True)
        E = model.layers[0].embeddings

        # concern with the subset of items
        T = tf.minimum(tf.shape(gru_output)[1], tf.shape(targets)[1])
        h = gru_output[:, :T, :]      # (B, T, d)
        y = targets[:, :T]           # (B, T)

        h_flat = tf.reshape(h, [-1, tf.shape(h)[-1]])   # (N, d), N = B*T
        y_flat = tf.reshape(y, [-1])                    # (N,)

        unique_items, y_new = tf.unique(y_flat)

        E_sub = tf.gather(E, unique_items)              # (K, d)

        # calculate the logits
        logits = tf.matmul(h_flat, E_sub, transpose_b=True)

        loss = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                y_new, logits, from_logits=True
            )
        )

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

    return loss


In [ ]:
epochs = 5
start = time.perf_counter()
model = get_model()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
optimizer.build(model.trainable_variables)

for epoch in range(epochs):
    total_loss = 0.0
    total_test_loss = 0.0
    step_count = 0

    for inputs, targets in train_ds:
        loss = train_step(inputs, targets, model, optimizer)
        total_loss += loss
        step_count += 1


    print(f"Epoch {epoch+1}/{epochs}, train_loss = {total_loss/step_count:.4f}")

tf.keras.backend.clear_session()

print("Total Time:", time.perf_counter() - start)

In [ ]:
model.save('/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model.keras')
print("Model has been saved")

Model has been saved


c) You should now evaluate your trained model on the held out test set that you prepared in question 1. In addition to computing the categorical cross entropy loss, the following standard metrics should also be computed:

* **Recall@k**: Measures the fraction of test cases where the true next item appears in the top-$k$ retrieved items. $$\text{Recall@k} = \frac{1}{N}\sum_{i=1}^N \mathbf{1}(\text{true}_i\in\text{top-}k_i)$$
* **MRR@k (mean reciprocal rank)**: Evaluates how high the true item ranks among the top-k predictions by averaging the reciprocal of its rank position. $$\text{MRR@k} = \frac{1}{N}\sum_{i=1}^N \frac{1}{\text{rank}_i} \mathbf{1}(\text{rank}_i\le k)$$
* **NDCG@k (normalised discounted cumulative gain)**: Weights correct hits higher when they appear earlier in the ranked list, normalized by the ideal ranking (in our case the ideal ranking is simply the one where the true item is at the top of the list). $$\text{NDCG@k} = \frac{1}{N}\sum_{i=1}^N \frac{1 / \log_2 (1 + \text{rank}_i)}{\text{IDCG@k}}$$

In the above equations, $N$ is the total number of next-item predictions across all examples in the test set, $i$ indexes the next-item predictions, $\text{true}_i$ is the index of the true next item, $\text{top-}k_i$ is the set of top-$k$ item indices ordered by logits/probabilities, $\mathbf{1}$ is the indicator function, and $\text{rank}_i$ is the rank of the true item in the top-$k$ predictions (with $\text{rank}_i=1$ being the top ranking). In each case (including the categorical cross entropy loss), the evaluation should be computed using all of the items present in the entire dataset for each next-item prediction $i$. In practice, you will compute each of the metrics for each batch and average across all batches. The recall@k, MRR@k and NDCG@k metrics should each be computed for $k=5, 20$.

Print out the metric results for your model.

**(5 marks)**

In [19]:
def get_test_loss(dataset, model):
  for inputs, targets in dataset:
        gru_output = model(inputs, training=True)
        E = model.layers[0].embeddings

        T = tf.minimum(tf.shape(gru_output)[1], tf.shape(targets)[1])
        h = gru_output[:, :T, :]
        y = targets[:, :T]

        h_flat = tf.reshape(h, [-1, tf.shape(h)[-1]])
        y_flat = tf.reshape(y, [-1])

        unique_items, y_new = tf.unique(y_flat)

        E_sub = tf.gather(E, unique_items)

        logits = tf.matmul(h_flat, E_sub, transpose_b=True)

        loss = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                y_new, logits, from_logits=True
            )
        )
        return loss

test_loss = get_test_loss(test_ds, model)
print(f"Test Loss: {test_loss:.4f}")



Test Loss: 4.1915


In [20]:
def recall_at_k(dataset, model, k):
  embeddings = model.layers[0].embeddings

  total_sample = 0
  total_hitting = 0
  for inputs, targets in dataset:
        gru_output = model(inputs, training=False)
        logits = tf.matmul(gru_output, embeddings, transpose_b=True)
        _ , topk = tf.math.top_k(logits, k=k)

        flat_targets = tf.reshape(targets, [-1]) # N *
        flat_topk = tf.reshape(topk, [-1, tf.shape(topk)[-1]]) # N * k

        #filter the padding
        idx = tf.squeeze(tf.where(flat_targets), axis=-1)
        flat_targets = tf.gather(flat_targets, idx)
        flat_topk = tf.gather(flat_topk, idx, axis=0)

        is_in_topk = tf.equal(tf.expand_dims(flat_targets,axis=-1), flat_topk)

        row_has_true = tf.reduce_any(is_in_topk, axis=-1)

        total_sample += tf.shape(idx)[0]
        total_hitting += tf.reduce_sum(tf.cast(row_has_true, tf.int32))


  recall = total_hitting / total_sample
  return recall.numpy()


In [21]:
def mrr_at_k(dataset, model, k):
  embeddings = model.layers[0].embeddings

  total_sample = 0.0
  total_rank = 0.0
  for inputs, targets in dataset:
        gru_output = model(inputs, training=False)
        logits = tf.matmul(gru_output, embeddings, transpose_b=True)
        #sorted_logits = tf.argsort(logits, axis=-1, direction='DESCENDING')
        _ , topk = tf.math.top_k(logits, k=k)

        flat_targets = tf.reshape(targets, [-1])
        flat_topk = tf.reshape(topk, [-1, tf.shape(topk)[-1]])

        #filter the padding
        idx = tf.squeeze(tf.where(flat_targets), axis=-1)
        flat_targets = tf.gather(flat_targets, idx)
        flat_topk = tf.gather(flat_topk, idx, axis=0)
        #print(flat_sorted_logits)
        rank = tf.where(tf.equal(flat_topk, tf.expand_dims(flat_targets,axis=-1)))
        rank = tf.cast(rank[:, 1], tf.float32)
        is_less_than_k = rank <= k
        #print(is_less_than_k)
        result = tf.cast(is_less_than_k, tf.float32)/(rank + 1.)

        total_sample += tf.cast(tf.shape(idx)[0], tf.float32)
        total_rank += tf.reduce_sum(result)

  mrr = total_rank / total_sample
  return mrr.numpy()




In [22]:
def ndcg_at_k(dataset, model, k):
  embeddings = model.layers[0].embeddings
  total_sample = 0.0
  total_rank = 0.0

  for inputs, targets in dataset:
        gru_output = model(inputs, training=False)
        logits = tf.matmul(gru_output, embeddings, transpose_b=True)
        _ , topk = tf.math.top_k(logits, k=k)

        flat_targets = tf.reshape(targets, [-1])
        flat_topk = tf.reshape(topk, [-1, tf.shape(topk)[-1]])

        idx = tf.squeeze(tf.where(flat_targets), axis=-1)
        flat_targets = tf.gather(flat_targets, idx)
        flat_topk = tf.gather(flat_topk, idx, axis=0)

        rank = tf.where(tf.equal(flat_topk, tf.expand_dims(flat_targets,axis=-1)))
        rank = tf.cast(rank[:, 1], tf.float32)
        is_less_than_k = rank < k

        result = tf.cast(is_less_than_k, tf.float32)*1. / tf.math.log(rank + 2.) * tf.math.log(tf.constant(2.0, dtype=tf.float32))

        total_sample += tf.cast(tf.shape(idx)[0], tf.float32)
        total_rank +=tf.reduce_sum(result)

  ndcg = total_rank / total_sample
  return ndcg.numpy()


In [44]:
from keras.models import load_model

model = load_model('/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model.keras')

model.summary()
print("Model loaded successfully!")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, None, 64)       │     2,931,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, None, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, None, 64)       │        24,960 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,956,800 (11.28 MB)

 Trainable params: 2,956,800 (11.28 MB)

 Non-trainable params: 0 (0.00 B)

Model loaded successfully!


In [45]:
print(f"Train loss is {get_test_loss(train_ds, model)}")
print(f"Test loss is {get_test_loss(test_ds, model)}")
print(f"k=5, metric recall@k is {recall_at_k(test_ds, model, 5)}, mrr@k is {mrr_at_k(test_ds, model, 5)}, ndcg@k is {ndcg_at_k(test_ds, model, 5)}")
print(f"k=20, metric recall@k is {recall_at_k(test_ds, model, 20)}, mrr@k is {mrr_at_k(test_ds, model, 20)}, ndcg@k is {ndcg_at_k(test_ds, model, 20)}")

Train loss is 4.348546981811523
Test loss is 4.129068374633789
k=5, metric recall@k is 0.32636729278102583, mrr@k is 0.19325441122055054, ndcg@k is 0.22618511319160461
k=20, metric recall@k is 0.5635029232491603, mrr@k is 0.2172442376613617, ndcg@k is 0.29442930221557617


### Question 3 (Total 10 marks)

In this question you should design a set of experiments to explore the effect of certain hyperparameters on the model's performance in the given task. It is up to you to decide what the hyperparameters and experiments should be; you can consider experimenting with various aspects of the model and training algorithm. Your goal should be to improve on the benchmark model developed in question 2.

The code below should be structured in such a way that executing the notebook would run all experiments successfully. Any custom code that is required should be provided, and the set of experiments should be clear from your code presentation.

It is not necessary for you to include the output for every training run that you do. Only the training and evaluation output for your best model should be displayed below.

**(10 marks)**

In [38]:
train_ds_256 = get_dataset_slices(train_sequences, batch_size = 256)
test_ds_256 = get_dataset_slices(test_sequences, batch_size = 256)

In [49]:
keras.backend.clear_session()

In [48]:
from keras.layers import Bidirectional

def get_model_bidirectional_gru():
  model = Sequential()
  model.add(Input(shape=(None,)))
  model.add(Embedding(input_dim=num_classes, output_dim=64, mask_zero=True))
  model.add(Dropout(0.1))
  model.add(Bidirectional(GRU(32, return_sequences=True)))

  return model

In [50]:
epochs = 5
start = time.perf_counter()

tf.keras.backend.clear_session()
opt_bidi = tf.keras.optimizers.Adam(learning_rate=1e-3)
model_bidi_gru = get_model_bidirectional_gru()
model_bidi_gru.summary()

opt_bidi.build(model_bidi_gru.trainable_variables)

for epoch in range(epochs):
  total_loss = 0.0
  step_count = 0

  for inputs, targets in train_ds_256:
      loss = train_step(inputs, targets, model_bidi_gru, opt_bidi)
      total_loss += loss
      step_count += 1.

  print(f"Bidirectional GRU Model - Epoch {epoch+1}/{epochs}, train_loss = {total_loss/step_count:.4f}")

print(f"\nBidirectional GRU Model Evaluation:")
print(f"k=5, metric recall@k is {recall_at_k(test_ds_256, model_bidi_gru, 5)}, mrr@k is {mrr_at_k(test_ds_256, model_bidi_gru, 5)}, ndcg@k is {ndcg_at_k(test_ds_256, model_bidi_gru, 5)}")
print(f"k=20, metric recall@k is {recall_at_k(test_ds_256, model_bidi_gru, 20)}, mrr@k is {mrr_at_k(test_ds_256, model_bidi_gru, 20)}, ndcg@k is {ndcg_at_k(test_ds_256, model_bidi_gru, 20)}")
print(f"The test loss of bidirctional model is {get_test_loss(test_ds_256, model_bidi_gru)}")

model_bidi_gru.save('/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model_bidirectional_gru.keras')
print("Bidirectional GRU Model has been saved")

print("Total Time for Bidirectional GRU Experiment:", time.perf_counter() - start)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 64)       │     2,931,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, None, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, None, 64)       │        18,816 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,950,656 (11.26 MB)

 Trainable params: 2,950,656 (11.26 MB)

 Non-trainable params: 0 (0.00 B)

Bidirectional GRU Model - Epoch 1/5, train_loss = 4.8844
Bidirectional GRU Model - Epoch 2/5, train_loss = 4.7341
Bidirectional GRU Model - Epoch 3/5, train_loss = 4.7158
Bidirectional GRU Model - Epoch 4/5, train_loss = 4.7081
Bidirectional GRU Model - Epoch 5/5, train_loss = 4.7036

Bidirectional GRU Model Evaluation:
k=5, metric recall@k is 0.7414272090226811, mrr@k is 0.6765491366386414, ndcg@k is 0.6927174925804138
k=20, metric recall@k is 0.8228635402413236, mrr@k is 0.6849142909049988, ndcg@k is 0.716289758682251
The test loss of bidirctional model is 4.550088882446289
Bidirectional GRU Model has been saved
Total Time for Bidirectional GRU Experiment: 477.9947512840008


### Question 4 (Total 10 marks)

Provide a separate PDF report with an account of the data processing, and the experiments you have run in question 3. Your report should include details of the choices you have made throughout the development of the models, including any reasoning or intuition behind the choices where appropriate.

Your report should also include the results from your experiments, highlighting the best choice of hyperparameters that you found. If you encountered difficulties to successfully train the model(s) you should report these, and if possible suggest what you think might be potential reasons for these difficulties. Are there additional experiments or modifications to the model and/or training algorithm that you would have liked to make given more time? What do you think are the most promising directions for performance gains?

Marks will be awarded for presentation and clarity. Your report should be no more than 2 pages, excluding references.

**(10 marks)**

**Appendix:**

In [ ]:
# Experiments with different optimizers
import time

epochs = 5
start = time.perf_counter()
rate = [1e-5, 1e-4]
for r in rate:
  tf.keras.backend.clear_session()
  opt = tf.keras.optimizers.Adam(learning_rate=r)
  model = get_model()
  for epoch in range(epochs):
    opt.build(model.trainable_variables)
    total_loss = 0.0
    total_test_loss = 0.0
    step_count = 0

    for inputs, targets in train_ds:
        loss = train_step(inputs, targets, model, opt)
        total_loss += loss
        step_count += 1.

    print(f"Rate = {r}, epoch {epoch+1}/{epochs}, train_loss = {total_loss/step_count:.4f}")

  print(f"k=5, metric recall@k is {recall_at_k(test_ds, model, 5)}, mrr@k is {mrr_at_k(test_ds, model, 5)}, ndcg@k is {ndcg_at_k(test_ds, model, 5)}")
  print(f"k=20, metric recall@k is {recall_at_k(test_ds, model, 20)}, mrr@k is {mrr_at_k(test_ds, model, 20)}, ndcg@k is {ndcg_at_k(test_ds, model, 20)}")

  model.save(f'/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model_{r}.keras')
  print("Model has been saved")

print("Total Time:", time.perf_counter() - start)

Rate = 1e-05, epoch 1/5, train_loss = 4.7024
Rate = 1e-05, epoch 2/5, train_loss = 4.6021
Rate = 1e-05, epoch 3/5, train_loss = 4.5390
Rate = 1e-05, epoch 4/5, train_loss = 4.4688
Rate = 1e-05, epoch 5/5, train_loss = 4.4098
k=5, metric recall@k is 0.12194717419247833, mrr@k is 0.06698454916477203, ndcg@k is 0.08066485822200775
k=20, metric recall@k is 0.2027200729775677, mrr@k is 0.07536153495311737, ndcg@k is 0.10413440316915512
Model has been saved
Rate = 0.0001, epoch 1/5, train_loss = 4.3455
Rate = 0.0001, epoch 2/5, train_loss = 4.1936
Rate = 0.0001, epoch 3/5, train_loss = 4.1617
Rate = 0.0001, epoch 4/5, train_loss = 4.1483
Rate = 0.0001, epoch 5/5, train_loss = 4.1408
k=5, metric recall@k is 0.3333747978604304, mrr@k is 0.2022487372159958, ndcg@k is 0.23483873903751373
k=20, metric recall@k is 0.5374632002322014, mrr@k is 0.22341907024383545, ndcg@k is 0.29417505860328674
Model has been saved
Total Time: 2574.518231743001


In [28]:
rate = [1e-5, 1e-4]
for r in rate:
  path = f'/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model_{r}.keras'

  model = tf.keras.models.load_model(path)
  print(f"When learning rate is {r}, the test loss is {get_test_loss(test_ds, model).numpy()}")

When learning rate is 1e-05, the test loss is 4.465348243713379
When learning rate is 0.0001, the test loss is 4.299195289611816


In [ ]:
def get_model_out(out_dim):

  model = Sequential()
  model.add(Input(shape=(None,)))
  model.add(Embedding(input_dim=num_classes, output_dim=out_dim, mask_zero=True))
  model.add(Dropout(0.1))
  model.add(GRU(out_dim, return_sequences=True))

  return model

In [ ]:
epochs = 5
start = time.perf_counter()
out = [128, 256]
for d in out:
  tf.keras.backend.clear_session()
  opt = tf.keras.optimizers.Adam(learning_rate=1e-3)
  model = get_model_out(d)
  for epoch in range(epochs):
    opt.build(model.trainable_variables)
    total_loss = 0.0
    total_test_loss = 0.0
    step_count = 0

    for inputs, targets in train_ds:
        loss = train_step(inputs, targets, model, opt)
        total_loss += loss
        step_count += 1.

    print(f"Out_dim = {d}, epoch {epoch+1}/{epochs}, train_loss = {total_loss/step_count:.4f}")

  print(f"k=5, metric recall@k is {recall_at_k(test_ds, model, 5)}, mrr@k is {mrr_at_k(test_ds, model, 5)}, ndcg@k is {ndcg_at_k(test_ds, model, 5)}")
  print(f"k=20, metric recall@k is {recall_at_k(test_ds, model, 20)}, mrr@k is {mrr_at_k(test_ds, model, 20)}, ndcg@k is {ndcg_at_k(test_ds, model, 20)}")

  model.save(f'/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model_outdim{d}.keras')
  print("Model has been saved")

print("Total Time:", time.perf_counter() - start)

Out_dim = 128, epoch 1/5, train_loss = 4.1275
Out_dim = 128, epoch 2/5, train_loss = 4.0918
Out_dim = 128, epoch 3/5, train_loss = 4.0829
Out_dim = 128, epoch 4/5, train_loss = 4.0783
Out_dim = 128, epoch 5/5, train_loss = 4.0756
k=5, metric recall@k is 0.3237964921010076, mrr@k is 0.1897236555814743, ndcg@k is 0.22293680906295776
k=20, metric recall@k is 0.5463780735580711, mrr@k is 0.21234777569770813, ndcg@k is 0.2871204614639282
Model has been saved
Out_dim = 256, epoch 1/5, train_loss = 4.1160
Out_dim = 256, epoch 2/5, train_loss = 4.0846
Out_dim = 256, epoch 3/5, train_loss = 4.0759
Out_dim = 256, epoch 4/5, train_loss = 4.0720
Out_dim = 256, epoch 5/5, train_loss = 4.0698
k=5, metric recall@k is 0.29978853091180496, mrr@k is 0.1737038493156433, ndcg@k is 0.2049655318260193
k=20, metric recall@k is 0.5264751005514782, mrr@k is 0.19703656435012817, ndcg@k is 0.2706393599510193
Model has been saved
Total Time: 4896.560447129001


In [32]:
out = [128, 256]
for d in out:
  path = f'/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model_outdim{d}.keras'
  model = tf.keras.models.load_model(path)
  print(f"When out_dim is {d}, the test loss is {get_test_loss(test_ds, model).numpy()}")

When out_dim is 128, the test loss is 4.061834335327148
When out_dim is 256, the test loss is 4.1891045570373535


In [ ]:
def get_model_dropout(drop):

  model = Sequential()
  model.add(Input(shape=(None,)))
  model.add(Embedding(input_dim=num_classes, output_dim=64, mask_zero=True))
  model.add(Dropout(drop))
  model.add(GRU(64, return_sequences=True))

  return model

In [ ]:
epochs = 5
start = time.perf_counter()
out = [0.1, 0.2, 0.3]
for d in out:
  tf.keras.backend.clear_session()
  opt = tf.keras.optimizers.Adam(learning_rate=1e-3)
  model = get_model_dropout(d)
  for epoch in range(epochs):
    opt.build(model.trainable_variables)
    total_loss = 0.0
    total_test_loss = 0.0
    step_count = 0

    for inputs, targets in train_ds:
        loss = train_step(inputs, targets, model, opt)
        total_loss += loss
        step_count += 1.

    print(f"dropout_rate = {d}, epoch {epoch+1}/{epochs}, train_loss = {total_loss/step_count:.4f}")

  print(f"k=5, metric recall@k is {recall_at_k(test_ds, model, 5)}, mrr@k is {mrr_at_k(test_ds, model, 5)}, ndcg@k is {ndcg_at_k(test_ds, model, 5)}")
  print(f"k=20, metric recall@k is {recall_at_k(test_ds, model, 20)}, mrr@k is {mrr_at_k(test_ds, model, 20)}, ndcg@k is {ndcg_at_k(test_ds, model, 20)}")

  model.save(f'/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model_dropout_rate{d}.keras')
  print("Model has been saved")

print("Total Time:", time.perf_counter() - start)

dropout_rate = 0.1, epoch 1/5, train_loss = 4.1497
dropout_rate = 0.1, epoch 2/5, train_loss = 4.1072
dropout_rate = 0.1, epoch 3/5, train_loss = 4.0981
dropout_rate = 0.1, epoch 4/5, train_loss = 4.0938
dropout_rate = 0.1, epoch 5/5, train_loss = 4.0913
k=5, metric recall@k is 0.30953269477961604, mrr@k is 0.1832655966281891, ndcg@k is 0.2145080268383026
k=20, metric recall@k is 0.5342704316457271, mrr@k is 0.20618879795074463, ndcg@k is 0.2794032096862793
Model has been saved
dropout_rate = 0.2, epoch 1/5, train_loss = 4.1560
dropout_rate = 0.2, epoch 2/5, train_loss = 4.1153
dropout_rate = 0.2, epoch 3/5, train_loss = 4.1072
dropout_rate = 0.2, epoch 4/5, train_loss = 4.1030
dropout_rate = 0.2, epoch 5/5, train_loss = 4.1009
k=5, metric recall@k is 0.3181158518887092, mrr@k is 0.18450260162353516, ndcg@k is 0.21756461262702942
k=20, metric recall@k is 0.5506489198490692, mrr@k is 0.20854826271533966, ndcg@k is 0.28505784273147583
Model has been saved
dropout_rate = 0.3, epoch 1/5, t

In [29]:
drop = [0.2, 0.3]
for d in drop:
  path = f'/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model_dropout_rate{d}.keras'
  model = tf.keras.models.load_model(path)
  print(f"When dropout rate is {d}, the test loss is {get_test_loss(test_ds, model).numpy()}")


When dropout rate is 0.2, the test loss is 4.152456283569336
When dropout rate is 0.3, the test loss is 4.088522434234619


In [ ]:
from keras.layers import LSTM

def get_model_lstm():
  model = Sequential()
  model.add(Input(shape=(None,)))
  model.add(Embedding(input_dim=num_classes, output_dim=64, mask_zero=True))
  model.add(Dropout(0.1))
  model.add(LSTM(64, return_sequences=True))

  return model


In [ ]:
start = time.perf_counter()
epochs = 5
model_lstm = get_model_lstm()
opt_lstm = tf.keras.optimizers.Adam(learning_rate=1e-3)

for epoch in range(epochs):
    total_loss = 0.0
    total_test_loss = 0.0
    step_count = 0

    for inputs, targets in train_ds:
        loss = train_step(inputs, targets, model_lstm, opt_lstm)
        total_loss += loss
        step_count += 1


    print(f"Epoch {epoch+1}/{epochs}, train_loss = {total_loss/step_count:.4f}")

print(f"k=5, metric recall@k is {recall_at_k(test_ds, model_lstm, 5)}, mrr@k is {mrr_at_k(test_ds, model_lstm, 5)}, ndcg@k is {ndcg_at_k(test_ds, model_lstm, 5)}")
print(f"k=20, metric recall@k is {recall_at_k(test_ds, model_lstm, 20)}, mrr@k is {mrr_at_k(test_ds, model_lstm, 20)}, ndcg@k is {ndcg_at_k(test_ds, model_lstm, 20)}")

tf.keras.backend.clear_session()

print("Total Time:", time.perf_counter() - start)

model.save("/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model_lstm.keras")

Epoch 1/5, train_loss = 4.1555
Epoch 2/5, train_loss = 4.1102
Epoch 3/5, train_loss = 4.1008
Epoch 4/5, train_loss = 4.0962
Epoch 5/5, train_loss = 4.0933
k=5, metric recall@k is 0.3281088029191027, mrr@k is 0.19587492942810059, ndcg@k is 0.22852009534835815
k=20, metric recall@k is 0.553012397893602, mrr@k is 0.21880553662776947, ndcg@k is 0.2934403419494629
Total Time: 1284.9166588199987


In [30]:
path = '/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model_lstm.keras'
model = keras.models.load_model(path)
print(f"Test loss is {get_test_loss(test_ds, model)}")

Test loss is 4.072671413421631


In [ ]:
from keras.layers import LSTM

def get_model_mixed():
  model = Sequential()
  model.add(Input(shape=(None,)))
  model.add(Embedding(input_dim=num_classes, output_dim=64, mask_zero=True))
  model.add(Dropout(0.1))
  model.add(GRU(64, return_sequences=True))
  model.add(LSTM(64, return_sequences=True))

  return model

In [ ]:
start = time.perf_counter()
model_mix = get_model_mixed()
opt_mix = tf.keras.optimizers.Adam(learning_rate=1e-3)
epochs = 5

for epoch in range(epochs):
    total_loss = 0.0
    total_test_loss = 0.0
    step_count = 0

    for inputs, targets in train_ds:
        loss = train_step(inputs, targets, model_mix, opt_mix)
        total_loss += loss
        step_count += 1



    print(f"Epoch {epoch+1}/{epochs}, train_loss = {total_loss/step_count:.4f}")

print(f"k=5, metric recall@k is {recall_at_k(test_ds, model_mix, 5)}, mrr@k is {mrr_at_k(test_ds, model_mix, 5)}, ndcg@k is {ndcg_at_k(test_ds, model_mix, 5)}")
print(f"k=20, metric recall@k is {recall_at_k(test_ds, model_mix, 20)}, mrr@k is {mrr_at_k(test_ds, model_mix, 20)}, ndcg@k is {ndcg_at_k(test_ds, model_mix, 20)}")


tf.keras.backend.clear_session()

print("Total Time:", time.perf_counter() - start)

model.save("/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model_mixed.keras")

Epoch 1/5, train_loss = 4.1638
Epoch 2/5, train_loss = 4.1141
Epoch 3/5, train_loss = 4.1036
Epoch 4/5, train_loss = 4.0982
Epoch 5/5, train_loss = 4.0951
k=5, metric recall@k is 0.3123522826222167, mrr@k is 0.18675412237644196, ndcg@k is 0.21791675686836243
k=20, metric recall@k is 0.545714641124518, mrr@k is 0.21059201657772064, ndcg@k is 0.2853448987007141
Total Time: 1750.8398979610001


In [31]:
path = '/content/drive/MyDrive/Imperial College London/Deep_Learning/Coursework 1/cw1_model_mixed.keras'
model = keras.models.load_model(path)
print(f"Test loss is {get_test_loss(test_ds, model)}")

Test loss is 4.795815467834473


In [33]:
def get_best_model():
  model = Sequential()
  model.add(Input(shape=(None,)))
  model.add(Embedding(input_dim=num_classes, output_dim=64, mask_zero=True))
  model.add(Dropout(0.1))
  model.add(GRU(64, return_sequences=True))

  return model

In [41]:
import time

epochs = 5
model_best = get_best_model()
opt = tf.keras.optimizers.Adam(learning_rate=1e-3)
opt.build(model_best.trainable_variables)
start = time.perf_counter()

for epoch in range(epochs):
    total_loss = 0.0
    total_test_loss = 0.0
    step_count = 0

    for inputs, targets in train_ds_256:
        loss = train_step(inputs, targets, model_best, opt)
        total_loss += loss
        step_count += 1.

    print(f"Epoch {epoch+1}/{epochs}, train_loss = {total_loss/step_count:.4f}")

tf.keras.backend.clear_session()

print("Total Time:", time.perf_counter() - start)

Epoch 1/5, train_loss = 5.2213
Epoch 2/5, train_loss = 5.1498
Epoch 3/5, train_loss = 5.1372
Epoch 4/5, train_loss = 5.1303
Epoch 5/5, train_loss = 5.1259
Total Time: 330.1726837059996


In [43]:
print(f"Test loss is {get_test_loss(test_ds_256, model_best)}")
print(f"k=5, metric recall@k is {recall_at_k(test_ds_256, model_best, 5)}, mrr@k is {mrr_at_k(test_ds, model_best, 5)}, ndcg@k is {ndcg_at_k(test_ds_256, model_best, 5)}")
print(f"k=20, metric recall@k is {recall_at_k(test_ds_256, model_best, 20)}, mrr@k is {mrr_at_k(test_ds, model_best, 20)}, ndcg@k is {ndcg_at_k(test_ds_256, model_best, 20)}")

Test loss is 5.085926532745361
k=5, metric recall@k is 0.38400298544595096, mrr@k is 0.23343485593795776, ndcg@k is 0.2707825005054474
k=20, metric recall@k is 0.6134676784011278, mrr@k is 0.25726938247680664, ndcg@k is 0.33752647042274475
